In [65]:
import pandas as pd
import numpy as np
import torch


In [66]:
train_df = pd.read_parquet("data/train.parquet")
test_df = pd.read_parquet("data/test.parquet")
val_df = pd.read_parquet("data/val.parquet")

In [67]:
train_df.head(5)

,pubid,question,context,long_answer,final_decision,combined_context
616,9347843,Laparoscopic-assisted ileocolic resections in ...,{'contexts': ['Because of the inflammatory nat...,The laparoscopic-assisted approach to Crohn's ...,no,Because of the inflammatory nature of Crohn's ...
684,21789019,Do elderly cancer patients have different care...,{'contexts': ['The increasingly older populati...,Elderly patients have informational and relati...,no,The increasingly older population confronts on...
208,26452334,Measurement of head and neck paragangliomas: i...,{'contexts': ['The aim of this study was to as...,"Due to a relatively good reproducibility, fast...",yes,The aim of this study was to assess the reprod...
952,23517744,Is solitary kidney really more resistant to is...,{'contexts': ['To our knowledge there are no e...,Solitary kidney in a canine model is more resi...,yes,To our knowledge there are no evidence-based m...
988,11438275,Does patient position during liver surgery inf...,{'contexts': ['It is generally believed that p...,The effect on venous pressures caused by the c...,no,It is generally believed that positioning of t...


In [68]:
train_df["Question and Context"]  = train_df["question"] + train_df["combined_context"]
test_df["Question and Context"]  = test_df["question"] + test_df["combined_context"]
val_df["Question and Context"]  = val_df["question"] + val_df["combined_context"]




In [69]:
train_df["Question and Context"].shape

(800,)

In [70]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X_tfidf =  vectorizer.fit_transform(train_df["Question and Context"])

In [71]:
X_tfidf_test = vectorizer.transform(test_df["Question and Context"])
X_tfidf_val = vectorizer.transform(val_df["Question and Context"])


In [72]:
!pip install gensim

In [73]:
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

tokenized_data = train_df["Question and Context"].apply(word_tokenize)
tokenized_data_test = test_df["Question and Context"].apply(word_tokenize)
tokenized_data_val = val_df["Question and Context"].apply(word_tokenize)

[nltk_data] Downloading package punkt to C:\Users\Aman Kumar
[nltk_data]     Singh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [74]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=tokenized_data,
    vector_size=100,   # embedding dimension
    window=5,          # context size
    min_count=2,       # ignore rare words
    workers=4,         # parallel processing
    sg=1               # 1 = Skip-gram, 0 = CBOW
)

In [75]:
!pip install --upgrade gensim numpy

  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)


In [76]:
def document_vector(doc):
    temp_list = [model.wv[i] for i in doc if i in model.wv]
    doc_vec = np.mean(temp_list, axis = 0)
    return doc_vec

In [77]:
tokenized_data

616    [Laparoscopic-assisted, ileocolic, resections,...
684    [Do, elderly, cancer, patients, have, differen...
208    [Measurement, of, head, and, neck, paraganglio...
952    [Is, solitary, kidney, really, more, resistant...
988    [Does, patient, position, during, liver, surge...
                             ...                        
918    [Should, displaced, midshaft, clavicular, frac...
418    [Evaluation, of, pediatric, VCUG, at, an, acad...
65     [Should, circumcision, be, performed, in, chil...
159    [Regional, anesthesia, as, compared, with, gen...
509    [Are, polymorphisms, in, oestrogen, receptors,...
Name: Question and Context, Length: 800, dtype: object

In [78]:
X_wv = tokenized_data.apply(document_vector)


In [79]:
x_wv_test = tokenized_data_test.apply(document_vector)
x_wv_val = tokenized_data_val.apply(document_vector)

In [80]:
from transformers import AutoTokenizer, AutoModel
MODEL_NAME = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

model.eval()
def document(text):
    encoded = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )
    
    with torch.no_grad():
        outputs = model(**encoded)

    cls_embedding = outputs.last_hidden_state[:, 0, :]

    return cls_embedding.squeeze().numpy()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [81]:
X_bert = train_df["Question and Context"].apply(document)
X_bert_test = test_df["Question and Context"].apply(document)
X_bert_val = val_df["Question and Context"].apply(document)




In [82]:
X_bert[1].shape

(768,)

In [83]:
X_tfidf.shape

(800, 12347)

In [84]:
X_wv[1].shape

(100,)

In [85]:
y_label = train_df["final_decision"]
y_label_test = test_df["final_decision"]
y_label_val = val_df["final_decision"]


In [99]:
np.savez_compressed(
    "embeddings_pubmedqa_train.npz",
    X_tfidf=X_tfidf.toarray(),
    X_wv=np.vstack(X_wv),
    X_bert=np.vstack(X_bert),
    y=y_label.values  # or y if already numpy
)

In [100]:
np.savez_compressed(
    "embeddings_pubmedqa_test.npz",
    X_tfidf_test=X_tfidf_test.toarray(),
    X_wv_test = np.vstack(x_wv_test),
    X_bert_test= np.vstack(X_bert_test),
    y=y_label_test.values  # or y if already numpy
)

In [101]:
np.savez_compressed(
    "embeddings_pubmedqa_val.npz",
    X_tfidf_val=X_tfidf_val.toarray(),
    X_wv_val = np.vstack(x_wv_val),
    X_bert_val= np.vstack(X_bert_val),
    y=y_label_val.values  # or y if already numpy
)

In [89]:
X_tfidf.shape

(800, 12347)

In [91]:
X_tfidf.shape

(800, 12347)